In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    max,
    min,
    avg,
    when,
    lit,
    current_timestamp
)

print("Gold notebook started")

In [0]:
silver_tables = [
    "silver_patient",
    "silver_encounter",
    "silver_observation",
    "silver_condition"
]

for table in silver_tables:

    table_name = f"workspace.fhir.{table}"

    print(
        f"{table_name}: "
        f"{spark.table(table_name).count()} records"
    )

In [0]:
patient_df = spark.table(
    "workspace.fhir.silver_patient"
)

encounter_df = spark.table(
    "workspace.fhir.silver_encounter"
)

observation_df = spark.table(
    "workspace.fhir.silver_observation"
)

condition_df = spark.table(
    "workspace.fhir.silver_condition"
)

In [0]:
patient_current_df = (
    patient_df
    .filter(col("is_current") == True)
)

encounter_current_df = (
    encounter_df
    .filter(col("is_current") == True)
)

observation_current_df = (
    observation_df
    .filter(col("is_current") == True)
)

condition_current_df = (
    condition_df
    .filter(col("is_current") == True)
)

In [0]:
gold_patient_df = (
    patient_current_df
    .select(
        "patient_id",
        "first_name",
        "last_name",
        "gender",
        "birth_date",
        "city",
        "state",
        "postal_code"
    )
)

In [0]:
display(gold_patient_df)

In [0]:
display(gold_patient_df)

In [0]:
(
    gold_patient_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.fhir.gold_patient"
    )
)

print("Gold Patient table created.")

In [0]:
display(
    spark.table("workspace.fhir.gold_patient")
)

In [0]:
from pyspark.sql.functions import regexp_replace

encounter_join_df = (
    encounter_current_df
    .withColumn(
        "patient_id",
        regexp_replace(
            col("patient_reference"),
            "^Patient/",
            ""
        )
    )
)

In [0]:
patient_encounter_df = (
    patient_current_df.alias("p")
    .join(
        encounter_join_df.alias("e"),
        col("p.patient_id") == col("e.patient_id"),
        "left"
    )
    .select(
        col("p.patient_id"),
        col("p.first_name"),
        col("p.last_name"),
        col("p.gender"),
        col("p.birth_date"),
        col("p.city"),
        col("p.state"),
        col("e.encounter_id"),
        col("e.status").alias("encounter_status"),
        col("e.class_code"),
        col("e.period_start"),
        col("e.period_end")
    )
)

display(patient_encounter_df)

In [0]:
(
    patient_encounter_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.fhir.gold_patient_encounters"
    )
)

print(
    "Gold Patient Encounter table created."
)

In [0]:
display(
    spark.table(
        "workspace.fhir.gold_patient_encounters"
    )
)

In [0]:
encounter_counts = (
    encounter_join_df
    .groupBy("patient_id")
    .agg(
        countDistinct("encounter_id")
        .alias("total_encounters")
    )
)

In [0]:
observation_join_df = (
    observation_current_df
    .withColumn(
        "patient_id",
        regexp_replace(
            col("patient_reference"),
            "^Patient/",
            ""
        )
    )
)

In [0]:
observation_counts = (
    observation_join_df
    .groupBy("patient_id")
    .agg(
        countDistinct("observation_id")
        .alias("total_observations")
    )
)

In [0]:
condition_join_df = (
    condition_current_df
    .withColumn(
        "patient_id",
        regexp_replace(
            col("patient_reference"),
            "^Patient/",
            ""
        )
    )
)

In [0]:
condition_counts = (
    condition_join_df
    .groupBy("patient_id")
    .agg(
        countDistinct("condition_id")
        .alias("total_conditions")
    )
)

In [0]:
gold_patient_activity_df = (
    patient_current_df.alias("p")

    .join(
        encounter_counts.alias("e"),
        col("p.patient_id") == col("e.patient_id"),
        "left"
    )

    .join(
        observation_counts.alias("o"),
        col("p.patient_id") == col("o.patient_id"),
        "left"
    )

    .join(
        condition_counts.alias("c"),
        col("p.patient_id") == col("c.patient_id"),
        "left"
    )

    .select(
        col("p.patient_id"),
        col("p.first_name"),
        col("p.last_name"),
        col("p.gender"),
        col("p.birth_date"),
        col("p.city"),
        col("p.state"),

        when(
            col("e.total_encounters").isNull(),
            lit(0)
        ).otherwise(
            col("e.total_encounters")
        ).alias("total_encounters"),

        when(
            col("o.total_observations").isNull(),
            lit(0)
        ).otherwise(
            col("o.total_observations")
        ).alias("total_observations"),

        when(
            col("c.total_conditions").isNull(),
            lit(0)
        ).otherwise(
            col("c.total_conditions")
        ).alias("total_conditions"),

        current_timestamp().alias(
            "gold_load_timestamp"
        )
    )
)

In [0]:
display(gold_patient_activity_df)

In [0]:
(
    gold_patient_activity_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.fhir.gold_patient_activity"
    )
)

print(
    "Gold Patient Activity table created."
)

In [0]:
gold_tables = [
    "gold_patient",
    "gold_patient_encounters",
    "gold_patient_activity"
]

for table in gold_tables:

    table_name = f"workspace.fhir.{table}"

    print(
        f"{table_name}: "
        f"{spark.table(table_name).count()} records"
    )

In [0]:
spark.sql("""
SHOW TABLES IN workspace.fhir
""").show(truncate=False)

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW workspace.fhir.vw_patient_activity
AS
SELECT
    patient_id,
    first_name,
    last_name,
    gender,
    birth_date,
    city,
    state,
    total_encounters,
    total_observations,
    total_conditions,
    gold_load_timestamp
FROM workspace.fhir.gold_patient_activity
""")

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.fhir.vw_patient_activity
        LIMIT 20
    """)
)

In [0]:
print("Gold Patient:")
print(
    spark.table(
        "workspace.fhir.gold_patient"
    ).count()
)

print("Gold Patient Encounters:")
print(
    spark.table(
        "workspace.fhir.gold_patient_encounters"
    ).count()
)

print("Gold Patient Activity:")
print(
    spark.table(
        "workspace.fhir.gold_patient_activity"
    ).count()
)

In [0]:
gold_tables = [
    "gold_patient",
    "gold_patient_encounters",
    "gold_patient_activity"
]

for table in gold_tables:
    table_name = f"workspace.fhir.{table}"
    print(
        f"{table_name}: "
        f"{spark.table(table_name).count()} records"
    )

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.fhir.vw_patient_activity
        LIMIT 20
    """)
)